# nb29 - Oracle bound: perfect per-cell subtraction on overlay windows

nb28 seed-0 came back flat (base_real 0.0502, mix_frac 0.0512): fraction supervision did not transfer. Before tuning the transfer (DANN, lambda, window), decide whether the target is even worth chasing: on overlay windows we KNOW the true per-cell photon fraction, so we can measure the resolution of a PERFECT subtractor - calibrate sum_i f_i e_i against Etrue. This is the ceiling of any fraction-based method at a given window size. If the ceiling sits near 0.03 the path is right and only the learning/transfer needs work; if the ceiling is ~0.05 the per-cell-fraction route cannot reach 0.03 at these window sizes and we pivot.

Estimators compared per window size W (half-width 2 -> 5x5, 3 -> 7x7): calibrated raw sum (anchor), oracle continuous frac sum, oracle binary mask (keep cells with f>0.5), and calibrated CLEAN-photon-only sum (sum of the true signal cells = removes pileup AND keeps leakage, the true floor of the window).

In [1]:
import os, sys, glob, time, pathlib
import numpy as np, pandas as pd, uproot, awkward as ak, matplotlib.pyplot as plt
REPO = pathlib.Path(os.environ['REPO_DIR']) if os.environ.get('REPO_DIR') else (
    pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd())
sys.path.insert(0, str(REPO / 'scripts'))
from run_experiments import split, resolution, PITCH
CLEAN = sorted((REPO / 'data' / 'full').glob('matched_*.root'))
MB = sorted((REPO / 'data' / 'minimum_bias').glob('*.root'))
MODE = os.environ.get('NB29_MODE', 'full')
if MODE == 'smoke': CLEAN, MB = CLEAN[:4], MB[:8]
THRESH = 2.49
print(len(CLEAN), 'clean files,', len(MB), 'minbias files | THRESH', THRESH, 'MeV')

100 clean files, 94 minbias files | THRESH 2.49 MeV


In [2]:
TK = ['cell_x','cell_y','energy','imodx','jmody']
AUX = ['sig_flux_prod_vertex_z','sig_flux_eTot']
def event_geom(cc):
    x, yy, e = cc['cell_x'], cc['cell_y'], cc['energy']
    ix, iy = cc['imodx'], cc['jmody']
    seed = int(np.argmax(e))
    pts = np.stack([x, yy], 1)
    pitch = np.full(len(x), np.nan)
    for key in {(int(p), int(q)) for p, q in zip(ix, iy)}:
        sel = (ix == key[0]) & (iy == key[1]); p = pts[sel]
        if len(p) >= 2:
            d = np.sqrt(((p[:, None, :] - p[None, :, :]) ** 2).sum(-1)); d[d == 0] = np.inf
            pitch[sel] = np.median(np.min(d, axis=1))
    fill = np.nanmedian(pitch) if np.isfinite(pitch).any() else 120.0
    pitch[~np.isfinite(pitch)] = fill
    ps = pitch[seed]
    ei = (x - x[seed]) / ps; ej = (yy - yy[seed]) / ps
    di = np.round(ei).astype(int); dj = np.round(ej).astype(int)
    ok = (np.abs(ei - di) < 0.15) & (np.abs(ej - dj) < 0.15)
    return seed, ps, di, dj, ok
def build_grid(files, keep_cheby, label):
    EV = []
    for path in files:
        with uproot.open(path) as f:
            a = f['clusters_matched'].arrays(TK + AUX, library='ak')
        vz = ak.to_numpy(a['sig_flux_prod_vertex_z']).astype(float)
        et_all = ak.to_numpy(a['sig_flux_eTot']).astype(float)
        for i in np.flatnonzero((vz < 100.0) & (et_all >= 1.0) & (et_all <= 100.0)):
            cc = {k: np.asarray(ak.to_numpy(a[k][i])).astype(float) for k in TK}
            e = cc['energy']
            if len(e) < 3: continue
            seed, ps, di, dj, ok = event_geom(cc)
            if ok.mean() < 0.5: continue
            ch = np.maximum(np.abs(di), np.abs(dj))
            keep = ok if keep_cheby is None else (ok & (ch <= keep_cheby))
            if keep.sum() < 1 or not keep[seed]: continue
            EV.append(dict(di=di[keep].astype(np.int16), dj=dj[keep].astype(np.int16),
                           e=e[keep].astype(np.float64), reg=int(np.argmin(np.abs(PITCH - ps))),
                           rmax=int(ch[ok].max()), Etrue=float(et_all[i])))
    print(f'{label}: {len(EV)} events')
    return EV
t0 = time.time()
CE = build_grid(CLEAN, 3, 'clean')
ME = build_grid(MB, None, 'minbias')
print(f'build {time.time()-t0:.0f}s')

clean: 30303 events


minbias: 72554 events
build 265s


In [3]:
def real_ratios(W):
    out = []
    for ev in ME:
        m = (np.maximum(np.abs(ev['di']), np.abs(ev['dj'])) <= W) & (ev['e'] >= THRESH)
        if m.sum(): out.append(ev['e'][m].sum() / (ev['Etrue'] * 1e3))
    return np.array(out)
def make_overlays(W, max_ovr=40000, seed=0):
    rng = np.random.default_rng(seed)
    rtr = split(len(ME))[0]
    pool_by_reg = {}
    need = 2 * W + 4
    for i in rtr:
        ev = ME[i]
        if ev['rmax'] >= need: pool_by_reg.setdefault(ev['reg'], []).append(ev)
    ratios = real_ratios(W)
    rows = []; dropped = 0
    for idx in rng.permutation(len(CE)):
        if len(rows) >= max_ovr: break
        A = CE[idx]
        pool = pool_by_reg.get(A['reg'])
        if not pool: continue
        pos = {}
        for k in range(len(A['e'])):
            if max(abs(A['di'][k]), abs(A['dj'][k])) <= W:
                pos[(int(A['di'][k]), int(A['dj'][k]))] = [float(A['e'][k]), float(A['e'][k])]
        if not pos: continue
        tgt = float(ratios[int(rng.integers(len(ratios)))])
        et_mev = A['Etrue'] * 1e3
        nd = 0
        while nd < 4 and sum(p[0] for p in pos.values()) / et_mev < tgt:
            B = pool[int(rng.integers(len(pool)))]
            di0, dj0 = 0, 0
            while max(abs(di0), abs(dj0)) < W + 2:
                di0 = int(rng.integers(-(W + 3), W + 4)); dj0 = int(rng.integers(-(W + 3), W + 4))
            for k in range(len(B['e'])):
                ri, rj = int(B['di'][k]) - di0, int(B['dj'][k]) - dj0
                if max(abs(ri), abs(rj)) > W: continue
                if (ri, rj) in pos: pos[(ri, rj)][0] += float(B['e'][k])
                else: pos[(ri, rj)] = [float(B['e'][k]), 0.0]
            nd += 1
        keys = list(pos.keys())
        V = np.array([pos[k] for k in keys])
        di = np.array([k[0] for k in keys]); dj = np.array([k[1] for k in keys])
        if not (di[np.argmax(V[:, 0])] == 0 and dj[np.argmax(V[:, 0])] == 0): dropped += 1; continue
        m = V[:, 0] >= THRESH
        if m.sum() < 1: continue
        e, sig = V[m, 0], V[m, 1]
        f = sig / e
        rows.append((float(e.sum()), float((f * e).sum()), float(e[f > 0.5].sum()),
                     float(sig.sum()), A['Etrue'], nd))
    R = np.array(rows)
    print(f'W={W}: {len(R)} overlays (dropped seed-displaced {dropped}), donors mean {R[:, 5].mean():.2f}, '
          f'ratio med {np.median(R[:, 0] / (R[:, 4] * 1e3)):.2f} (real {np.median(ratios):.2f})')
    return R

In [4]:
def calib_sigma(x, Etrue, tr_frac=0.5, per_bin=False):
    n = len(x); idx = np.random.default_rng(1).permutation(n)
    tr, te = idx[:int(tr_frac * n)], idx[int(tr_frac * n):]
    lx = np.log(np.clip(x, 1e-3, None)); ly = np.log(Etrue)
    a, b = np.polyfit(lx[tr], ly[tr], 1)
    pe = np.exp(a * lx[te] + b)
    out = {'overall': resolution(pe, Etrue[te])['sigma_eff']}
    if per_bin:
        edges = np.quantile(Etrue[te], np.linspace(0, 1, 7))
        for i in range(6):
            hi = edges[i + 1] + (1e-9 if i == 5 else 0)
            mm = (Etrue[te] >= edges[i]) & (Etrue[te] < hi)
            out[f'{edges[i]:.0f}-{edges[i+1]:.0f}'] = resolution(pe[mm], Etrue[te][mm])['sigma_eff']
    return out
RES = {}
for W in (2, 3):
    R = make_overlays(W)
    Et = R[:, 4]
    RES[W] = {
        'raw sum (anchor)': calib_sigma(R[:, 0], Et, per_bin=True),
        'oracle frac sum': calib_sigma(R[:, 1], Et, per_bin=True),
        'oracle binary mask': calib_sigma(R[:, 2], Et, per_bin=True),
        'true signal sum (floor)': calib_sigma(R[:, 3], Et, per_bin=True),
    }
    print(f'--- W={W} ({2*W+1}x{2*W+1}) overlay test ---')
    for k, v in RES[W].items():
        print(f'  {k:26s} overall {v["overall"]:.4f} | per-bin ' +
              ' '.join(f'{kk}:{vv:.3f}' for kk, vv in v.items() if kk != 'overall'))

W=2: 17116 overlays (dropped seed-displaced 1151), donors mean 2.40, ratio med 1.36 (real 1.18)
--- W=2 (5x5) overlay test ---
  raw sum (anchor)           overall 0.2452 | per-bin 1-17:0.446 17-27:0.291 27-37:0.207 37-49:0.178 49-67:0.163 67-100:0.142
  oracle frac sum            overall 0.0840 | per-bin 1-17:0.068 17-27:0.041 27-37:0.035 37-49:0.032 49-67:0.030 67-100:0.033
  oracle binary mask         overall 0.2440 | per-bin 1-17:0.239 17-27:0.087 27-37:0.059 37-49:0.048 49-67:0.045 67-100:0.048
  true signal sum (floor)    overall 0.0840 | per-bin 1-17:0.068 17-27:0.041 27-37:0.035 37-49:0.032 49-67:0.030 67-100:0.033


W=3: 8885 overlays (dropped seed-displaced 832), donors mean 2.29, ratio med 1.77 (real 1.38)
--- W=3 (7x7) overlay test ---
  raw sum (anchor)           overall 0.3051 | per-bin 1-25:0.603 25-39:0.350 39-51:0.271 51-64:0.199 64-79:0.171 79-100:0.146
  oracle frac sum            overall 0.1061 | per-bin 1-25:0.121 25-39:0.057 39-51:0.037 51-64:0.031 64-79:0.028 79-100:0.026
  oracle binary mask         overall 0.2604 | per-bin 1-25:0.389 25-39:0.113 39-51:0.061 51-64:0.043 64-79:0.038 79-100:0.036
  true signal sum (floor)    overall 0.1061 | per-bin 1-25:0.121 25-39:0.057 39-51:0.037 51-64:0.031 64-79:0.028 79-100:0.026


## Verdict
**Correction (post-audit):** `oracle frac sum` and `true signal sum` are algebraically the SAME quantity (f_i * e_i = signal_i by construction), so the table shows them identical - the earlier draft wrongly narrated them as two different estimators. What this study actually measures is the calibrated-sum resolution of the PURE clean shower content of a fixed window, i.e. the containment/estimator floor of nb16 re-derived on overlay statistics - an upper bound on fraction-based *sum readouts*, not on models. Two further caveats from the audit: the overlay contamination distribution overshoots the real one (ratio med 1.35 vs 1.18), which inflates the raw-sum anchor rather than making the bound conservative, and the seed-displacement drop (events where pileup out-shines the photon are discarded) biases the oracle slightly optimistic. The per-bin conclusion stands in weakened form: even PERFECT per-cell separation with a sum readout cannot beat the window containment spread, ~0.030-0.041 per bin at E>17 GeV at 5x5; a model readout on top (globals, leakage correction) is needed to do better.

In [5]:
print('anchors: minbias model best 0.0459 (kNN-81) | base_real nb28 0.0502 | clean per-bin floor 0.028-0.038 | goal 0.03')
for W in (2, 3):
    o = RES[W]
    print(f'W={W}: anchor {o["raw sum (anchor)"]["overall"]:.4f} -> oracle frac {o["oracle frac sum"]["overall"]:.4f} '
          f'-> true signal {o["true signal sum (floor)"]["overall"]:.4f}')

anchors: minbias model best 0.0459 (kNN-81) | base_real nb28 0.0502 | clean per-bin floor 0.028-0.038 | goal 0.03
W=2: anchor 0.2452 -> oracle frac 0.0840 -> true signal 0.0840
W=3: anchor 0.3051 -> oracle frac 0.1061 -> true signal 0.1061


## Data request rationale (for Felipe)
Chain of evidence ending here: (1) nb28 Phase 1 - the resolution spread is fully explained by pileup contamination; the cleanest events already reach 0.023-0.033. (2) This notebook - if per-cell photon fractions were KNOWN, a plain calibrated sum recovers 0.030-0.041 per bin at E > 17 GeV (5x5 window), against 0.037-0.049 for our best trained models. (3) nb28 Phases 3-4 and nb30 - fractions manufactured from overlay synthesis do NOT transfer (domain gap AUC 0.74); nb31 - in-situ density context does not help either. Therefore the one input that unlocks the remaining 15-25% per bin is **per-cell truth from simulation**: for each cell of each min-bias cluster, the energy deposited by the signal photon vs by everything else (equivalently a per-cell signal fraction), as used by Belle II (arXiv:2306.04179, +30% resolution, per-crystal time the top feature - timing we already have). A per-photon column alongside the existing cell_energies arrays in the ntuples would be sufficient; no new reconstruction is needed on our side - the training pipeline (nb30) accepts these labels as-is.